# Phase 5: 손절/익절 전략 백테스트 엔진

## 이 노트북의 핵심
단순 예측 성능(RMSE)이 아닌, **실제 돈을 얼마 버느냐**를 측정합니다.

---

## 전략 구조 전체 그림

```
매일 장 시작 전:
  모델 실행 → pred_low, pred_high 확보
  ↓
  매수주문가  = pred_low × 0.995   (예측 저가보다 약간 낮게)
  손절가(SL)  = 매수가 - 1.0 × ATR  (ATR 기반 동적 손절)
  익절가(TP)  = pred_high × 0.99   (예측 고가 바로 아래)
  
  ── 진입 필터 ──
  Risk/Reward = (TP - 매수가) / (매수가 - SL)
  R/R < 1.5 이면 → 이날은 거래 안 함 (나쁜 셋업 걸러냄)
  
  ── 포지션 크기 ──
  1트레이드 최대 손실 = 전체 자본 × 2%
  주식 수 = (자본 × 2%) / (매수가 - SL)
  
장중:
  실제 저가 ≤ 매수주문가 → 매수 체결
  이후 실제 고가 ≥ TP   → 익절 체결  ✅
  이후 실제 저가 ≤ SL   → 손절 체결  ❌
  당일 둘 다 없으면      → 다음날 시가에 청산
  
트레일링 스탑:
  매수 후 +3% 되면 → 손절선을 매수가(본전)로 올림
  매수 후 +5% 되면 → 손절선을 현재 고점-2%로 트레일링
```

---

## 핵심 개념 설명

### ATR (Average True Range)
하루의 "실제" 가격 변동 범위 평균입니다.  
TSLL의 ATR이 3이면 → 하루 평균 $3 움직인다는 뜻.  
손절을 `매수가 - 1.0 × ATR`로 설정하면, **정상적인 일중 변동**에는 손절 안 걸리고  
진짜 추세가 반전됐을 때만 손절됩니다.

### Risk/Reward 필터
예: 매수가 $20, 손절가 $18, 익절가 $25  
Risk = $2, Reward = $5 → R/R = 2.5  
이 거래는 진입! (R/R > 1.5 기준 통과)  

예: 매수가 $20, 손절가 $18, 익절가 $21  
Risk = $2, Reward = $1 → R/R = 0.5  
이 거래는 스킵! (수익보다 손실이 크므로 진입 안 함)

### 고정 분할 리스크 (Fixed Fractional)
자본이 $100,000이고 리스크 2%면 → 최대 손실 = $2,000  
ATR이 $3이면 → 주식 수 = $2,000 / $3 = 666주  
자본이 줄어도, 늘어도 **항상 자본의 2%만 위험**에 노출됩니다.

In [ ]:
# 한글 폰트 설치 (없으면 차트에서 한글이 □□□로 깨집니다)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum', '-q'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.font_manager as fm
import itertools
import warnings
import os

warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 (Colab 필수) ──────────────────────────────
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
# ──────────────────────────────────────────────────────────────

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

# ─── 전략 기본 파라미터 ───────────────────────────────────────────
INITIAL_CAPITAL  = 100_000
RISK_PER_TRADE   = 0.02
SL_ATR_MULT      = 1.0
TP_PRED_MULT     = 0.99
BUY_PRED_MULT    = 0.995
MIN_RR_RATIO     = 1.5
MAX_HOLD_DAYS    = 2
COMMISSION       = 0.0005
TRAILING_STOP_PCT= 0.02
BREAKEVEN_PCT    = 0.03
# ──────────────────────────────────────────────────────────────────

print('전략 파라미터 설정 완료')
print(f'  - 리스크/트레이드: {RISK_PER_TRADE*100:.0f}%')
print(f'  - 손절 ATR 배수: {SL_ATR_MULT}x')
print(f'  - 최소 R/R 비율: {MIN_RR_RATIO}')
print(f'  - 트레일링 스탑: 고점 -{TRAILING_STOP_PCT*100:.0f}%')
print('한글 폰트: NanumGothic')

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 핵심 백테스트 엔진
# ───────────────────────────────────────────────────────────────────

def run_strategy(
    pred_df,
    model_name       = 'Model',
    capital          = INITIAL_CAPITAL,
    risk_per_trade   = RISK_PER_TRADE,
    sl_atr_mult      = SL_ATR_MULT,
    tp_pred_mult     = TP_PRED_MULT,
    buy_pred_mult    = BUY_PRED_MULT,
    min_rr           = MIN_RR_RATIO,
    max_hold         = MAX_HOLD_DAYS,
    trailing_stop    = TRAILING_STOP_PCT,
    breakeven_pct    = BREAKEVEN_PCT,
    verbose          = False
):
    """
    손절/익절/트레일링스탑/포지션사이징 통합 백테스트 엔진

    pred_df 필수 컬럼:
      date, pred_high, pred_low, actual_high, actual_low, actual_open, actual_close, atr_14
    """
    cash      = capital
    shares    = 0
    entry_px  = 0.0
    stop_loss = 0.0
    take_prof = 0.0
    hold_days = 0
    high_since_entry = 0.0

    portfolio_history = []
    trade_log         = []

    for i, row in pred_df.iterrows():
        date      = row['date']
        o, h, l, c = row['actual_open'], row['actual_high'], row['actual_low'], row['actual_close']
        atr       = row.get('atr_14', c * 0.07)  # ATR 없으면 가격의 7% 대용

        # ── 현재 포트폴리오 가치 기록 ──
        port_val = cash + shares * c
        portfolio_history.append({'date': date, 'value': port_val, 'cash': cash, 'shares': shares})

        if shares == 0:
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━
            # 포지션 없음 → 매수 시도
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━
            buy_price = row['pred_low'] * buy_pred_mult
            sl_price  = buy_price - (sl_atr_mult * atr)
            tp_price  = row['pred_high'] * tp_pred_mult

            # 음수 손절가 방지
            sl_price = max(sl_price, buy_price * 0.90)

            risk_per_share   = buy_price - sl_price
            reward_per_share = tp_price  - buy_price

            # Risk/Reward 필터
            rr_ratio = reward_per_share / risk_per_share if risk_per_share > 0 else 0
            if rr_ratio < min_rr:
                if verbose:
                    print(f'{date} | R/R={rr_ratio:.2f} 미달 → 스킵')
                continue

            # 오늘 저가가 매수주문가 이하면 체결
            if l <= buy_price:
                # 체결가: 매수주문가와 시가 중 불리한 쪽 (현실적)
                fill_price = max(buy_price, min(o, buy_price * 1.002))

                # 포지션 크기 계산 (고정 분할 리스크)
                dollar_risk = cash * risk_per_trade
                qty         = int(dollar_risk / risk_per_share)
                cost        = qty * fill_price * (1 + COMMISSION)

                if qty > 0 and cost <= cash:
                    cash    -= cost
                    shares   = qty
                    entry_px = fill_price
                    stop_loss = sl_price
                    take_prof = tp_price
                    hold_days = 0
                    high_since_entry = fill_price

                    trade_log.append({
                        'date': date, 'action': 'BUY',
                        'price': round(fill_price, 2),
                        'shares': qty,
                        'stop_loss': round(stop_loss, 2),
                        'take_profit': round(take_prof, 2),
                        'rr_ratio': round(rr_ratio, 2),
                        'capital': round(port_val, 0)
                    })
                    if verbose:
                        print(f'{date} | 매수 {qty}주 @ ${fill_price:.2f} | SL=${stop_loss:.2f} TP=${take_prof:.2f} R/R={rr_ratio:.2f}')

        else:
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━
            # 포지션 보유 중 → 청산 조건 체크
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━
            hold_days        += 1
            high_since_entry  = max(high_since_entry, h)
            pnl_pct           = (h - entry_px) / entry_px  # 최고 수익률

            # ── 트레일링 스탑 업데이트 ──
            if pnl_pct >= breakeven_pct:
                # 본전 이상이면 손절을 본전으로 올림 (원금 보호)
                stop_loss = max(stop_loss, entry_px)
            if pnl_pct >= 0.05:
                # +5% 이상이면 트레일링 스탑 활성화
                trailing_sl = high_since_entry * (1 - trailing_stop)
                stop_loss   = max(stop_loss, trailing_sl)

            exit_price  = None
            exit_reason = None

            # 우선순위 1: 손절 (저가가 손절선 이하)
            if l <= stop_loss:
                exit_price  = stop_loss
                exit_reason = '손절' if stop_loss < entry_px else '본전청산'

            # 우선순위 2: 익절 (고가가 익절선 이상)
            elif h >= take_prof:
                exit_price  = take_prof
                exit_reason = '익절'

            # 우선순위 3: 최대 보유일 초과 → 종가 청산
            elif hold_days >= max_hold:
                exit_price  = c
                exit_reason = f'{max_hold}일 보유 청산'

            if exit_price is not None:
                revenue  = shares * exit_price * (1 - COMMISSION)
                pnl      = revenue - (shares * entry_px * (1 + COMMISSION))
                pnl_pct_ = pnl / (shares * entry_px) * 100
                cash    += revenue

                trade_log.append({
                    'date': date, 'action': exit_reason,
                    'price': round(exit_price, 2),
                    'shares': shares,
                    'pnl': round(pnl, 2),
                    'pnl_pct': round(pnl_pct_, 2),
                    'hold_days': hold_days,
                    'capital': round(cash, 0)
                })
                if verbose:
                    emoji = '✅' if pnl > 0 else '❌'
                    print(f'{date} | {exit_reason} @ ${exit_price:.2f} | P&L: ${pnl:+.0f} ({pnl_pct_:+.1f}%) {emoji}')

                shares    = 0
                entry_px  = 0.0
                hold_days = 0

    # ── 최종 미청산 포지션 마감 ──
    if shares > 0:
        final_price = pred_df.iloc[-1]['actual_close']
        cash += shares * final_price * (1 - COMMISSION)

    # ── 성능 지표 계산 ──
    port_df       = pd.DataFrame(portfolio_history)
    port_df['date'] = pd.to_datetime(port_df['date'])
    port_df        = port_df.set_index('date')

    final_val     = cash
    total_ret     = (final_val - capital) / capital * 100
    daily_rets    = port_df['value'].pct_change().dropna()
    sharpe        = (daily_rets.mean() / daily_rets.std() * np.sqrt(252)) if daily_rets.std() > 0 else 0
    rolling_max   = port_df['value'].cummax()
    dd            = (port_df['value'] - rolling_max) / rolling_max * 100
    mdd           = dd.min()

    trades_df     = pd.DataFrame(trade_log)
    buy_cnt       = len(trades_df[trades_df['action'] == 'BUY']) if len(trades_df) > 0 else 0
    win_cnt       = len(trades_df[trades_df.get('pnl', pd.Series(dtype=float)) > 0]) if len(trades_df) > 0 else 0
    win_rate      = win_cnt / buy_cnt * 100 if buy_cnt > 0 else 0

    metrics = {
        'model':            model_name,
        'final_capital':    round(final_val, 0),
        'total_return(%)':  round(total_ret, 2),
        'sharpe_ratio':     round(sharpe, 3),
        'MDD(%)':           round(mdd, 2),
        'total_trades':     buy_cnt,
        'win_rate(%)':      round(win_rate, 1),
        'sl_atr_mult':      sl_atr_mult,
        'min_rr':           min_rr,
    }
    return port_df, trades_df, metrics

print('백테스트 엔진 로드 완료')

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 모델별 백테스트 실행
# ───────────────────────────────────────────────────────────────────

# 테스트 데이터에 ATR 포함되어 있어야 함
# 만약 없으면 Phase 2 특성 데이터에서 가져옴
test_features = pd.read_csv('/content/data/processed/test.csv', index_col=0, parse_dates=True)

model_files = {
    'ARIMA':   '/content/results/arima_predictions.csv',
    'Prophet': '/content/results/prophet_predictions.csv',
    'XGBoost': '/content/results/xgboost_predictions.csv',
    'LSTM':    '/content/results/lstm_predictions.csv',
}

all_port  = {}
all_trades= {}
all_metrics = []

for model_name, filepath in model_files.items():
    if not os.path.exists(filepath):
        print(f'{model_name}: 파일 없음, 스킵')
        continue

    pred = pd.read_csv(filepath)

    # ATR 합치기
    pred['date'] = pd.to_datetime(pred['date'])
    atr_map = test_features['atr_14'].to_dict()
    pred['atr_14'] = pred['date'].map(atr_map).fillna(pred['pred_low'] * 0.07)
    if 'actual_open' not in pred.columns:
        pred['actual_open'] = pred['actual_close']

    pred = pred.reset_index(drop=True)
    port, trades, metrics = run_strategy(pred, model_name, verbose=False)
    all_port[model_name]   = port
    all_trades[model_name] = trades
    all_metrics.append(metrics)

    print(f'[{model_name}] 수익률: {metrics["total_return(%)"]:.1f}% | '
          f'Sharpe: {metrics["sharpe_ratio"]:.2f} | '
          f'MDD: {metrics["MDD(%)"]:.1f}% | '
          f'거래: {metrics["total_trades"]}회 | '
          f'승률: {metrics["win_rate(%)"]:.0f}%')

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 그리드서치: 손절/R/R 파라미터 최적화
# 가장 많이 벌 수 있는 조합 찾기
# ───────────────────────────────────────────────────────────────────
print('그리드서치 중... (최적 손절/R/R 비율 탐색)')

# 최고 성능 모델 파일 사용
best_model_name = pd.DataFrame(all_metrics).sort_values('total_return(%)', ascending=False).iloc[0]['model']
best_file = model_files.get(best_model_name)

if best_file and os.path.exists(best_file):
    pred_base = pd.read_csv(best_file)
    pred_base['date'] = pd.to_datetime(pred_base['date'])
    atr_map = test_features['atr_14'].to_dict()
    pred_base['atr_14'] = pred_base['date'].map(atr_map).fillna(pred_base['pred_low'] * 0.07)
    if 'actual_open' not in pred_base.columns:
        pred_base['actual_open'] = pred_base['actual_close']
    pred_base = pred_base.reset_index(drop=True)

    sl_range = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
    rr_range = [1.0, 1.2, 1.5, 2.0, 2.5]

    grid_results = []
    for sl_mult, min_rr in itertools.product(sl_range, rr_range):
        _, _, m = run_strategy(
            pred_base, best_model_name,
            sl_atr_mult=sl_mult, min_rr=min_rr, verbose=False
        )
        grid_results.append({
            'sl_atr_mult': sl_mult,
            'min_rr': min_rr,
            'return(%)': m['total_return(%)'],
            'sharpe': m['sharpe_ratio'],
            'mdd(%)': m['MDD(%)'],
            'trades': m['total_trades'],
            'win_rate(%)': m['win_rate(%)']
        })

    grid_df = pd.DataFrame(grid_results)

    # 수익률 히트맵
    pivot = grid_df.pivot(index='sl_atr_mult', columns='min_rr', values='return(%)')
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    im1 = axes[0].imshow(pivot.values, cmap='RdYlGn', aspect='auto')
    plt.colorbar(im1, ax=axes[0])
    axes[0].set_xticks(range(len(rr_range)))
    axes[0].set_yticks(range(len(sl_range)))
    axes[0].set_xticklabels([f'R/R≥{r}' for r in rr_range])
    axes[0].set_yticklabels([f'SL={s}x ATR' for s in sl_range])
    for i in range(len(sl_range)):
        for j in range(len(rr_range)):
            axes[0].text(j, i, f'{pivot.values[i,j]:.1f}%',
                         ha='center', va='center', fontsize=8,
                         color='black' if abs(pivot.values[i,j]) < 30 else 'white')
    axes[0].set_title(f'[{best_model_name}] 수익률 그리드서치')
    axes[0].set_xlabel('최소 R/R 비율')
    axes[0].set_ylabel('손절 ATR 배수')

    # Sharpe 히트맵
    pivot_sh = grid_df.pivot(index='sl_atr_mult', columns='min_rr', values='sharpe')
    im2 = axes[1].imshow(pivot_sh.values, cmap='RdYlGn', aspect='auto')
    plt.colorbar(im2, ax=axes[1])
    axes[1].set_xticks(range(len(rr_range)))
    axes[1].set_yticks(range(len(sl_range)))
    axes[1].set_xticklabels([f'R/R≥{r}' for r in rr_range])
    axes[1].set_yticklabels([f'SL={s}x ATR' for s in sl_range])
    for i in range(len(sl_range)):
        for j in range(len(rr_range)):
            axes[1].text(j, i, f'{pivot_sh.values[i,j]:.2f}',
                         ha='center', va='center', fontsize=8)
    axes[1].set_title(f'[{best_model_name}] Sharpe Ratio 그리드서치')
    axes[1].set_xlabel('최소 R/R 비율')

    plt.tight_layout()
    plt.savefig('/content/gridsearch_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

    best_params = grid_df.sort_values('return(%)', ascending=False).iloc[0]
    print(f'\n최적 파라미터:')
    print(f'  손절 ATR 배수: {best_params["sl_atr_mult"]}x')
    print(f'  최소 R/R: {best_params["min_rr"]}')
    print(f'  수익률: {best_params["return(%)"]:.1f}%')
    print(f'  Sharpe: {best_params["sharpe"]:.2f}')

    grid_df.to_csv('/content/results/gridsearch_results.csv', index=False)

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 최고 모델 상세 분석 — 거래 내역 완전 해부
# ───────────────────────────────────────────────────────────────────

if all_metrics:
    best = pd.DataFrame(all_metrics).sort_values('total_return(%)', ascending=False).iloc[0]
    bn   = best['model']
    port = all_port[bn]
    trades = all_trades[bn]

    fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=False)

    # ① 포트폴리오 가치
    axes[0].plot(port.index, port['value'], color='#ffd700', lw=2)
    axes[0].axhline(INITIAL_CAPITAL, color='gray', ls='--', lw=1, label='시드 $100,000')
    axes[0].set_title(f'[{bn}] 포트폴리오 성장', fontsize=12)
    axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # ② MDD
    rolling_max = port['value'].cummax()
    dd = (port['value'] - rolling_max) / rolling_max * 100
    axes[1].fill_between(port.index, dd, 0, alpha=0.6, color='#ef5350')
    axes[1].axhline(-best['MDD(%)'], color='#ffd700', ls='--', lw=1,
                    label=f'MDD: {best["MDD(%)"]:.1f}%')
    axes[1].set_title('최대 낙폭 (MDD)', fontsize=12)
    axes[1].set_ylabel('낙폭 (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # ③ 거래별 P&L
    if len(trades) > 0 and 'pnl' in trades.columns:
        closed = trades.dropna(subset=['pnl'])
        colors = ['#26a69a' if v > 0 else '#ef5350' for v in closed['pnl']]
        axes[2].bar(range(len(closed)), closed['pnl'], color=colors, width=0.7)
        axes[2].axhline(0, color='white', lw=0.5)
        axes[2].set_title(f'거래별 P&L (총 {len(closed)}건, 승률 {best["win_rate(%)"]:.0f}%)', fontsize=12)
        axes[2].set_ylabel('수익/손실 ($)')
        axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:+,.0f}'))
        axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('/content/strategy_detail.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 거래 요약 통계
    if len(trades) > 0 and 'pnl' in trades.columns:
        closed = trades.dropna(subset=['pnl'])
        wins   = closed[closed['pnl'] > 0]
        losses = closed[closed['pnl'] <= 0]
        print('\n' + '='*55)
        print(f'[{bn}] 거래 통계 상세')
        print('='*55)
        print(f'총 거래:     {len(closed)}건')
        print(f'익절:        {len(wins)}건 ({len(wins)/len(closed)*100:.0f}%)')
        print(f'손절/청산:   {len(losses)}건 ({len(losses)/len(closed)*100:.0f}%)')
        if len(wins) > 0:
            print(f'평균 수익:   ${wins["pnl"].mean():,.0f} ({wins["pnl_pct"].mean():+.1f}%)')
        if len(losses) > 0:
            print(f'평균 손실:   ${losses["pnl"].mean():,.0f} ({losses["pnl_pct"].mean():+.1f}%)')
        if len(wins) > 0 and len(losses) > 0 and losses['pnl'].mean() != 0:
            profit_factor = wins['pnl'].sum() / abs(losses['pnl'].sum())
            print(f'Profit Factor: {profit_factor:.2f}  (1 이상이면 수익성 있음)')
        print(f'최종 자본:   ${best["final_capital"]:,.0f}')
        print(f'총 수익률:   {best["total_return(%)"]:.1f}%')
        print(f'Sharpe:      {best["sharpe_ratio"]:.2f}')
        print(f'MDD:         {best["MDD(%)"]:.1f}%')
        print('='*55)

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 페이퍼트레이딩 일일 신호 생성기
# 매일 아침 이 셀 실행 → 오늘의 매수가/손절가/익절가 출력
# ───────────────────────────────────────────────────────────────────
import yfinance as yf
import joblib
from datetime import datetime

def generate_daily_signal(model_type='xgboost'):
    """
    오늘의 트레이딩 신호 생성
    매일 장 열리기 전 실행하세요
    """
    print('='*60)
    print(f'TSLL 일일 트레이딩 신호 — {datetime.now().strftime("%Y-%m-%d")}')
    print('='*60)

    # 최신 데이터 수집
    tsll = yf.download('TSLL', period='100d', progress=False)
    tsll.columns = [col[0] if isinstance(col, tuple) else col for col in tsll.columns]

    if len(tsll) < 60:
        print('데이터 부족')
        return

    # ATR 계산
    import pandas_ta as ta
    tsll['atr_14'] = ta.atr(tsll['High'], tsll['Low'], tsll['Close'], length=14)

    latest = tsll.iloc[-1]
    atr = latest['atr_14'] if not pd.isna(latest['atr_14']) else latest['Close'] * 0.07
    close = latest['Close']

    # 모델 예측 (저장된 모델 사용)
    try:
        if model_type == 'xgboost':
            # 실제로는 피처 벡터 구성 필요 — 여기서는 ATR 기반 간단 추정
            xgb_h = joblib.load('/content/models/xgb_high.pkl')
            xgb_l = joblib.load('/content/models/xgb_low.pkl')
            # 피처 구성 (실제 운용 시 02번 노트북 add_features 함수 재사용)
            print('※ XGBoost 예측: 02번 노트북 피처 필요. 아래는 ATR 기반 추정값입니다.')
    except:
        pass

    # ATR 기반 단순 추정 (모델 없을 때 백업)
    pred_high = close * 1.035   # 현재가 +3.5% (ATR 기반 조정 필요)
    pred_low  = close * 0.967   # 현재가 -3.3%

    buy_price = pred_low  * BUY_PRED_MULT
    sl_price  = buy_price - (SL_ATR_MULT * atr)
    tp_price  = pred_high * TP_PRED_MULT

    risk   = buy_price - sl_price
    reward = tp_price  - buy_price
    rr     = reward / risk if risk > 0 else 0

    print(f'현재 종가:    ${close:.2f}')
    print(f'ATR(14):     ${atr:.2f} ({atr/close*100:.1f}%)')
    print(f'예측 고가:    ${pred_high:.2f}')
    print(f'예측 저가:    ${pred_low:.2f}')
    print()
    print(f'📌 매수주문가: ${buy_price:.2f}')
    print(f'🔴 손절가(SL): ${sl_price:.2f}  (손실 {(sl_price-buy_price)/buy_price*100:.1f}%)')
    print(f'🟢 익절가(TP): ${tp_price:.2f}  (수익 {(tp_price-buy_price)/buy_price*100:.1f}%)')
    print(f'R/R 비율:    {rr:.2f}  ({"✅ 진입 가능" if rr >= MIN_RR_RATIO else "❌ 진입 금지 (R/R 미달)"})')

    if rr >= MIN_RR_RATIO:
        position_size = int((INITIAL_CAPITAL * RISK_PER_TRADE) / risk)
        print()
        print(f'💰 권장 매수 수량: {position_size}주 (총 ${position_size*buy_price:,.0f})')
        print(f'   최대 손실 예상: ${position_size*risk:,.0f} (자본의 {position_size*risk/INITIAL_CAPITAL*100:.1f}%)')
    print('='*60)

# 실행
generate_daily_signal()

In [ ]:
# ───────────────────────────────────────────────────────────────────
# 페이퍼트레이딩 일지 (수동으로 채워넣기)
# ───────────────────────────────────────────────────────────────────
import json

# 아래 데이터를 매일 업데이트하세요
paper_trades = [
    # 예시 (실제 날짜와 가격으로 교체)
    # {
    #     'date': '2026-06-09',
    #     'pred_high': 25.50,
    #     'pred_low': 22.30,
    #     'buy_order': 22.19,    # pred_low * 0.995
    #     'sl': 20.65,           # buy - ATR
    #     'tp': 25.25,           # pred_high * 0.99
    #     'actual_high': 24.80,
    #     'actual_low': 22.10,
    #     'result': '익절',      # '익절' / '손절' / '미체결' / '강제청산'
    #     'exit_price': 25.25,
    #     'pnl_pct': 4.8,
    #     'note': '테슬라 실적 발표 영향'
    # },
]

if paper_trades:
    pt_df = pd.DataFrame(paper_trades)
    pt_df['date'] = pd.to_datetime(pt_df['date'])

    wins   = pt_df[pt_df['result'] == '익절']
    losses = pt_df[pt_df['result'] == '손절']

    print('페이퍼트레이딩 누적 성과')
    print(f'총 거래: {len(pt_df)}건 | 익절: {len(wins)} | 손절: {len(losses)}')
    if len(pt_df) > 0:
        print(f'승률: {len(wins)/len(pt_df)*100:.0f}%')
        print(pt_df[['date','buy_order','sl','tp','result','exit_price','pnl_pct']].to_string(index=False))
else:
    print('아직 페이퍼트레이딩 데이터 없음')
    print('3주차부터 매일 위 딕셔너리에 데이터 추가하세요')